# TBD：客群分群結合生存分析與回購策略

這份 Notebook 是由 `TBD_分群成果_客群分群結合生存分析與回購策略.md` 改寫而成，目的是讓討論時，可以直接看到模型計算流程與結果如何產生。

資料來源：`2023-2025_旅平險_backup.csv`  
分析定位：方法確認用，不是最終正式報告版。

## 本 Notebook 會產出

1. 分群特徵表。
2. K 值選擇表。
3. 各客群輪廓表。
4. 各客群回購曲線。
5. 各客群 30/60/120/180/365 天累積回購率。
6. 各客群提醒策略表。
7. 推薦主行銷對象與需要問助教的假設。

## 重要提醒

本分析採用「描述型分群」：使用 2023-2025 完整期間的投保行為，包含回購間隔、投保次數、目的地占比等特徵，回顧性地整理客戶樣態。  
因此，這不是「新客第一次投保當下即可預測未來客群」的即時預測模型。

## 1. 載入套件與設定路徑

本 Notebook 只使用 `pandas`、`numpy` 與 Python 標準函式庫。  
因為目前環境沒有 `sklearn` 與 `matplotlib`，所以 K-means 與 SVG 圖表會在 Notebook 中自行實作。

In [ ]:
from pathlib import Path
import html
import math
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# 讓 Notebook 可以從「期末報告」資料夾或「TBD」資料夾執行
cwd = Path.cwd()
if (cwd / '2023-2025_旅平險_backup.csv').exists():
    ROOT = cwd
elif (cwd.parent / '2023-2025_旅平險_backup.csv').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError('找不到 2023-2025_旅平險_backup.csv，請確認 Notebook 是否在期末報告資料夾或 TBD 資料夾內執行。')

DATA_PATH = ROOT / '2023-2025_旅平險_backup.csv'
OUT_DIR = ROOT / 'TBD'
OUT_DIR.mkdir(exist_ok=True)

DATA_PATH, OUT_DIR

## 2. 讀取原始資料並建立基礎欄位

原始資料是一列一張保單。  
後續分群會轉成「一列一位客戶」，客戶識別使用 `APL_ID`。

主要欄位處理：

- `LOGIN_DATE`：投保日期，用來排序與計算回購間隔。
- `EFF_DATE`：保單生效日，用來計算提前投保天數。
- `SITE_CODE`：旅遊目的地代碼，整理成目的地類型。
- `CARD_BANK = 999999`：視為非信用卡/特殊付款代碼。

In [ ]:
dtypes = {
    'NO': 'string',
    'APL_ID': 'string',
    'APL_BIRTH': 'string',
    'APL_ADD_CD': 'string',
    'SITE_CODE': 'string',
    'CARD_BANK': 'string',
}

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig', dtype=dtypes)

for col in ['LOGIN_DATE', 'EFF_DATE']:
    df[col + '_DT'] = pd.to_datetime(df[col].astype(str), format='%Y%m%d', errors='coerce')

df['LOGIN_YEAR'] = df['LOGIN_DATE_DT'].dt.year
df['AGE'] = df['LOGIN_YEAR'] - df['APL_BIRTH'].str[:4].astype(int)

def norm_site(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    return s.zfill(3) if s.isdigit() and len(s) <= 3 else s

def dest_cat(code):
    s = str(code) if pd.notna(code) else 'Missing'
    if s == '098':
        return 'domestic'
    if s == '131':
        return 'japan'
    if s == '132':
        return 'korea'
    if s in ['101', '141', '142']:
        return 'china_hk_macau'
    if s in ['140', '143', '144', '145', '146', '147', '148']:
        return 'seasia'
    if s in ['301', '302', '212', '510']:
        return 'longhaul'
    if len(s) > 3:
        return 'multi'
    if s == '999':
        return 'other_custom'
    return 'other'

df['SITE_NORM'] = df['SITE_CODE'].astype('string').str.strip().map(norm_site)
df['DEST_CAT'] = df['SITE_NORM'].map(dest_cat)
df['LEAD_DAYS'] = (df['EFF_DATE_DT'] - df['LOGIN_DATE_DT']).dt.days

print('資料列數：', f'{len(df):,}')
print('投保者數：', f'{df["APL_ID"].nunique():,}')
print('LOGIN_DATE 範圍：', df['LOGIN_DATE_DT'].min().date(), '到', df['LOGIN_DATE_DT'].max().date())
df.head()

## 3. 建立客戶層級資料與分群特徵

這一步會把保單層級資料整理成客戶層級資料。  
一次性客戶沒有回購間隔，所以先保留，但不放入 K-means 分群。

特徵分為五類：

1. 回購行為：投保次數、平均回購間隔、短週期回購占比等。
2. 價值行為：平均保費、總保費、平均保額、高保費占比。
3. 旅遊行為：平均天數、目的地數、日本占比、國內占比、長程占比等。
4. 購買行為：平均提前投保天數、當天投保占比、3 天內投保占比。
5. 人口特徵：首次投保年齡。

In [ ]:
high_prem_threshold = df['TOT_PREM'].quantile(0.75)

sdf = df.sort_values(['APL_ID', 'LOGIN_DATE_DT', 'EFF_DATE_DT', 'SITE_NORM', 'TOT_PREM']).copy()
sdf['ORDER'] = sdf.groupby('APL_ID').cumcount() + 1
sdf['PREV_LOGIN'] = sdf.groupby('APL_ID')['LOGIN_DATE_DT'].shift(1)
sdf['GAP'] = (sdf['LOGIN_DATE_DT'] - sdf['PREV_LOGIN']).dt.days

cust = sdf.groupby('APL_ID').agg(
    policies=('NO', 'size'),
    age_first=('AGE', 'first'),
    avg_age=('AGE', 'mean'),
    avg_prem=('TOT_PREM', 'mean'),
    total_prem=('TOT_PREM', 'sum'),
    avg_sa=('TOT_SA', 'mean'),
    high_prem_share=('TOT_PREM', lambda x: (x > high_prem_threshold).mean()),
    avg_days=('DAYS', 'mean'),
    long_days_share=('DAYS', lambda x: (x > 14).mean()),
    unique_dest=('SITE_NORM', lambda x: x.nunique(dropna=True)),
    japan_share=('DEST_CAT', lambda x: (x == 'japan').mean()),
    domestic_share=('DEST_CAT', lambda x: (x == 'domestic').mean()),
    seasia_share=('DEST_CAT', lambda x: (x == 'seasia').mean()),
    longhaul_share=('DEST_CAT', lambda x: (x == 'longhaul').mean()),
    multi_dest_share=('DEST_CAT', lambda x: (x == 'multi').mean()),
    avg_lead=('LEAD_DAYS', 'mean'),
    same_day_share=('LEAD_DAYS', lambda x: (x == 0).mean()),
    within3_share=('LEAD_DAYS', lambda x: (x <= 3).mean()),
    noncard_share=('CARD_BANK', lambda x: (x == '999999').mean()),
    first_login=('LOGIN_DATE_DT', 'first'),
    last_login=('LOGIN_DATE_DT', 'last'),
)

gaps = sdf[sdf['GAP'].notna()].groupby('APL_ID').agg(
    repeat_events=('GAP', 'size'),
    avg_gap=('GAP', 'mean'),
    median_gap=('GAP', 'median'),
    last_gap=('GAP', 'last'),
    gap_le30_share=('GAP', lambda x: (x <= 30).mean()),
    gap_le60_share=('GAP', lambda x: (x <= 60).mean()),
    gap_le120_share=('GAP', lambda x: (x <= 120).mean()),
)

cust = cust.join(gaps)
cust['is_repeat'] = cust['policies'] > 1

first = sdf[sdf['ORDER'] == 1].set_index('APL_ID')
second = sdf[sdf['ORDER'] == 2].set_index('APL_ID')
cust['first_to_second_days'] = (second['LOGIN_DATE_DT'] - first['LOGIN_DATE_DT']).dt.days

print('總客戶數：', f'{len(cust):,}')
print('重複投保者：', f'{cust["is_repeat"].sum():,}')
print('一次性/尚未回購客戶：', f'{(~cust["is_repeat"]).sum():,}')

cust.head()

## 4. 分群特徵表

下表是本次 K-means 使用的特徵。  
`repeat_events` 會列在特徵表中，但不另外放入模型，因為它與 `policies` 幾乎是同一件事，放入會重複加權。

In [ ]:
features = [
    'policies', 'avg_gap', 'median_gap', 'last_gap',
    'gap_le30_share', 'gap_le60_share', 'gap_le120_share',
    'avg_prem', 'total_prem', 'avg_sa', 'high_prem_share',
    'avg_days', 'long_days_share', 'unique_dest',
    'japan_share', 'domestic_share', 'seasia_share', 'longhaul_share', 'multi_dest_share',
    'avg_lead', 'same_day_share', 'within3_share', 'noncard_share',
    'age_first',
]

feature_labels = {
    'policies': '投保次數',
    'repeat_events': '回購事件數',
    'avg_gap': '平均回購間隔',
    'median_gap': '回購間隔中位數',
    'last_gap': '最近一次回購間隔',
    'gap_le30_share': '30天內回購占比',
    'gap_le60_share': '60天內回購占比',
    'gap_le120_share': '120天內回購占比',
    'avg_prem': '平均保費',
    'total_prem': '總保費',
    'avg_sa': '平均保額',
    'high_prem_share': '高保費保單占比',
    'avg_days': '平均保單天數',
    'long_days_share': '長天期保單占比',
    'unique_dest': '目的地數',
    'japan_share': '日本占比',
    'domestic_share': '國內占比',
    'seasia_share': '東南亞占比',
    'longhaul_share': '歐美澳長程占比',
    'multi_dest_share': '多目的地占比',
    'avg_lead': '平均提前投保天數',
    'same_day_share': '當天投保占比',
    'within3_share': '3天內投保占比',
    'noncard_share': '非信用卡/特殊付款占比',
    'age_first': '首次投保年齡',
}

log_cols = {
    'policies', 'avg_gap', 'median_gap', 'last_gap',
    'avg_prem', 'total_prem', 'avg_sa', 'avg_days',
    'unique_dest', 'avg_lead', 'age_first',
}

feature_table_rows = []
feature_groups = {
    '回購行為': ['policies', 'repeat_events', 'avg_gap', 'median_gap', 'last_gap', 'gap_le30_share', 'gap_le60_share', 'gap_le120_share'],
    '價值行為': ['avg_prem', 'total_prem', 'avg_sa', 'high_prem_share'],
    '旅遊行為': ['avg_days', 'long_days_share', 'unique_dest', 'japan_share', 'domestic_share', 'seasia_share', 'longhaul_share', 'multi_dest_share'],
    '購買行為': ['avg_lead', 'same_day_share', 'within3_share', 'noncard_share'],
    '人口特徵': ['age_first'],
}

for group, fs in feature_groups.items():
    for f in fs:
        if f == 'repeat_events':
            used = '否，避免與投保次數重複加權'
            process = '由投保次數衍生'
        else:
            used = '是'
            process = 'log1p 後標準化' if f in log_cols else '比例/數值標準化'
        feature_table_rows.append({
            '特徵類型': group,
            '特徵': feature_labels[f],
            '欄位名稱': f,
            '模型前處理': process,
            '是否放入 K-means': used,
        })

feature_table = pd.DataFrame(feature_table_rows)
feature_table

## 5. K-means 實作與特徵標準化

K-means 會受到特徵尺度影響。  
例如「總保費」數值可能到幾萬元，但「日本占比」只在 0 到 1 之間。若不標準化，總保費會主導分群。

本 Notebook 的處理方式：

1. 對偏態明顯的正數特徵做 `log1p`，降低極端值影響。
2. 對所有特徵做標準化，讓平均為 0、標準差為 1。
3. 使用自行實作的 K-means++ 初始化與 K-means 迭代。

In [ ]:
rep = cust[cust['is_repeat']].copy()

Xdf = rep[features].astype(float).copy()
for c in log_cols:
    Xdf[c] = np.log1p(Xdf[c])

means = Xdf.mean()
stds = Xdf.std(ddof=0).replace(0, 1)
X = ((Xdf - means) / stds).to_numpy(dtype=float)

print('K-means 分群對象：', f'{len(rep):,}', '位重複投保者')
print('模型特徵數：', len(features))

In [ ]:
def kmeans_pp(X, k, rng):
    n = X.shape[0]
    centers = np.empty((k, X.shape[1]))
    idx = rng.integers(n)
    centers[0] = X[idx]
    dist2 = np.sum((X - centers[0]) ** 2, axis=1)
    for i in range(1, k):
        probs = dist2 / dist2.sum()
        idx = rng.choice(n, p=probs)
        centers[i] = X[idx]
        dist2 = np.minimum(dist2, np.sum((X - centers[i]) ** 2, axis=1))
    return centers


def run_kmeans(X, k, seed, n_init=10, max_iter=100):
    best = None
    for init in range(n_init):
        rng = np.random.default_rng(seed + init * 100 + k)
        centers = kmeans_pp(X, k, rng)
        labels = None
        for it in range(max_iter):
            d = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
            new_labels = d.argmin(axis=1)
            if labels is not None and np.array_equal(new_labels, labels):
                break
            labels = new_labels
            for j in range(k):
                mask = labels == j
                centers[j] = X[mask].mean(axis=0) if mask.any() else X[rng.integers(X.shape[0])]
        inertia = float(((X - centers[labels]) ** 2).sum())
        if best is None or inertia < best[0]:
            best = (inertia, labels.copy(), centers.copy(), it + 1)
    return best


def sample_silhouette(X, labels, seed=42, sample_n=2500):
    # 抽樣估計輪廓係數，避免完整距離矩陣過大
    rng = np.random.default_rng(seed)
    n = len(labels)
    idx = rng.choice(n, size=min(sample_n, n), replace=False)
    Xs = X[idx]
    ls = labels[idx]
    D = np.sqrt(((Xs[:, None, :] - Xs[None, :, :]) ** 2).sum(axis=2))
    uniq = np.unique(ls)
    vals = []
    for i in range(len(idx)):
        own = ls == ls[i]
        a = D[i, own].sum() / (own.sum() - 1) if own.sum() > 1 else 0.0
        b = np.inf
        for lab in uniq:
            if lab == ls[i]:
                continue
            m = ls == lab
            if m.any():
                b = min(b, D[i, m].mean())
        vals.append((b - a) / max(a, b) if max(a, b) > 0 and np.isfinite(b) else 0.0)
    return float(np.mean(vals))

## 6. K 值選擇

本次測試 K=3 到 K=6。  
指標說明：

- 群內差異：越低越好，代表同一群內的客戶越相似。
- 抽樣輪廓係數：越高越好，代表群與群分得越清楚。
- 商務可解釋性：是否能命名成清楚客群。

目前暫定採用 K=5，原因是 K=3 雖然輪廓係數最高，但客群較粗；K=5 能拆出比較有意義的商務輪廓。這一點需要與助教確認。

In [ ]:
metrics = []
labels_by_k = {}
prev_inertia = None

for k in range(3, 7):
    inertia, labels, centers, iters = run_kmeans(X, k, seed=20260524, n_init=10, max_iter=100)
    sil = sample_silhouette(X, labels, seed=20260524 + k, sample_n=2500)
    improvement = None if prev_inertia is None else (prev_inertia - inertia) / prev_inertia * 100
    counts = pd.Series(labels).value_counts()
    metrics.append({
        'K': k,
        '群內差異': inertia,
        '每位客戶群內差異': inertia / len(X),
        '相較前一K改善%': improvement,
        '抽樣輪廓係數': sil,
        '最小群人數': counts.min(),
        '最大群人數': counts.max(),
    })
    labels_by_k[k] = labels
    prev_inertia = inertia

k_table = pd.DataFrame(metrics)
k_table

![K值選擇圖](K值選擇圖.svg)

## 7. 暫定 K=5 並建立客群名稱

K-means 只會產生群號，不會自動產生有商務意義的名稱。  
因此下列名稱是根據各群的平均投保次數、平均回購間隔、目的地占比、平均保費、平均天數等特徵人工命名。

In [ ]:
K_FINAL = 5
rep['cluster_raw'] = labels_by_k[K_FINAL]

name_map = {
    0: '長程高保費低頻型',
    1: '高頻短週期多目的地型',
    2: '低頻長週期日本型',
    3: '一般亞洲穩定回購型',
    4: '國內短天期低保費型',
}

rep['cluster_name'] = rep['cluster_raw'].map(name_map)

cust['cluster_name'] = '一次性/尚未回購客群'
cust.loc[rep.index, 'cluster_name'] = rep['cluster_name']

cluster_order = [
    '一次性/尚未回購客群',
    '高頻短週期多目的地型',
    '一般亞洲穩定回購型',
    '低頻長週期日本型',
    '長程高保費低頻型',
    '國內短天期低保費型',
]

cust['cluster_name'].value_counts().reindex(cluster_order)

## 8. 各客群輪廓表

這張表是目前最重要的成果之一。  
它用來確認每個客群是否真的有可解釋的行為輪廓。

In [ ]:
profile = cust.groupby('cluster_name').agg(
    客戶數=('policies', 'size'),
    保單數=('policies', 'sum'),
    平均投保次數=('policies', 'mean'),
    平均保費=('avg_prem', 'mean'),
    總保費=('total_prem', 'sum'),
    平均天數=('avg_days', 'mean'),
    平均回購間隔=('avg_gap', 'mean'),
    平均目的地數=('unique_dest', 'mean'),
    日本占比=('japan_share', 'mean'),
    國內占比=('domestic_share', 'mean'),
    長程占比=('longhaul_share', 'mean'),
    非信用卡占比=('noncard_share', 'mean'),
    首次投保年齡=('age_first', 'mean'),
).reindex(cluster_order)

profile['客戶占比'] = profile['客戶數'] / len(cust) * 100
profile['保費貢獻占比'] = profile['總保費'] / profile['總保費'].sum() * 100

profile_display = profile[[
    '客戶數', '客戶占比', '平均投保次數', '平均保費', '總保費', '保費貢獻占比',
    '平均天數', '平均回購間隔', '平均目的地數', '日本占比', '國內占比', '長程占比',
]].copy()

profile_display

![客群特徵熱圖](客群特徵熱圖.svg)

客群初步解讀：

- 高頻短週期多目的地型：平均投保次數高、回購間隔短、目的地數多。
- 一般亞洲穩定回購型：客群最大，保費貢獻高，適合作為主力經營基礎。
- 低頻長週期日本型：日本占比高，但回購週期長。
- 長程高保費低頻型：平均保費與平均天數最高，單次價值高但回購慢。
- 國內短天期低保費型：國內占比高、平均保費低、天數短。
- 一次性/尚未回購客群：只有一筆投保紀錄，不可直接判斷永遠不會回購。

## 9. 各客群 30/60/120/180/365 天累積回購率

這裡使用「第一次投保到第二次投保」的天數。  
注意：K-means 群組只包含已觀察到重複投保者，因此這些曲線比較的是「已回購客群之間的回購速度」。  
一次性/尚未回購客群的 0% 是定義結果，不等於真實永遠不回購。

In [ ]:
thresholds = [30, 60, 120, 180, 365]
curve_rows = []

for g in cluster_order:
    sub = cust[cust['cluster_name'] == g]
    row = {'客群': g, '客戶數': len(sub)}
    for t in thresholds:
        if g == '一次性/尚未回購客群':
            row[f'{t}天'] = 0.0
        else:
            row[f'{t}天'] = (sub['first_to_second_days'] <= t).mean() * 100
    ev = sub['first_to_second_days'].dropna()
    row['回購間隔P25'] = ev.quantile(0.25) if len(ev) else np.nan
    row['回購間隔中位數'] = ev.median() if len(ev) else np.nan
    row['回購間隔P75'] = ev.quantile(0.75) if len(ev) else np.nan
    curve_rows.append(row)

curve = pd.DataFrame(curve_rows).set_index('客群').reindex(cluster_order)
curve

![客群回購曲線](客群回購曲線.svg)

觀察重點：

- 高頻短週期多目的地型：60 天內已有約 59.0% 完成第二次投保，最適合短週期提醒。
- 一般亞洲穩定回購型：120 天內約 56.5% 完成第二次投保，客群規模最大，適合作為主行銷基礎。
- 低頻長週期日本型：180 天內約 30.0%，365 天約 64.4%，不是不回購，而是回購週期較長。
- 長程高保費低頻型：365 天約 77.8%，前期慢但單次價值高，適合延後提醒。

## 10. 產生 SVG 圖表

以下程式會輸出三張 SVG 圖到 `TBD` 資料夾：

1. `K值選擇圖.svg`
2. `客群特徵熱圖.svg`
3. `客群回購曲線.svg`

如果前面的模型重跑後結果變動，可以重新執行這格更新圖表。

In [ ]:
def svg_line_chart_k(metrics_df, path):
    W, H = 860, 420
    m = {'l': 70, 'r': 60, 't': 40, 'b': 60}
    plotW = W - m['l'] - m['r']
    plotH = H - m['t'] - m['b']
    ks = metrics_df['K'].tolist()
    y1 = metrics_df['每位客戶群內差異'].tolist()
    y2 = metrics_df['抽樣輪廓係數'].tolist()

    def sx(k):
        return m['l'] + (k - min(ks)) / (max(ks) - min(ks)) * plotW

    y1min, y1max = min(y1) * 0.98, max(y1) * 1.02
    y2min, y2max = 0, max(y2) * 1.2

    def sy1(v):
        return m['t'] + (y1max - v) / (y1max - y1min) * plotH

    def sy2(v):
        return m['t'] + (y2max - v) / (y2max - y2min) * plotH

    pts1 = ' '.join(f'{sx(k):.1f},{sy1(v):.1f}' for k, v in zip(ks, y1))
    pts2 = ' '.join(f'{sx(k):.1f},{sy2(v):.1f}' for k, v in zip(ks, y2))

    s = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">', '<rect width="100%" height="100%" fill="white"/>']
    s.append('<style>text{font-family:-apple-system,BlinkMacSystemFont,"Noto Sans TC",Arial,sans-serif;font-size:13px;fill:#222}.title{font-size:20px;font-weight:700}.axis{stroke:#444;stroke-width:1}.grid{stroke:#ddd;stroke-width:1}</style>')
    s.append(f'<text x="{W/2}" y="24" text-anchor="middle" class="title">K 值選擇：群內差異與輪廓係數</text>')
    for i in range(5):
        y = m['t'] + i * plotH / 4
        s.append(f'<line x1="{m["l"]}" y1="{y}" x2="{W-m["r"]}" y2="{y}" class="grid"/>')
    s.append(f'<line x1="{m["l"]}" y1="{m["t"]}" x2="{m["l"]}" y2="{H-m["b"]}" class="axis"/>')
    s.append(f'<line x1="{m["l"]}" y1="{H-m["b"]}" x2="{W-m["r"]}" y2="{H-m["b"]}" class="axis"/>')
    s.append(f'<line x1="{W-m["r"]}" y1="{m["t"]}" x2="{W-m["r"]}" y2="{H-m["b"]}" class="axis"/>')
    for k in ks:
        x = sx(k)
        s.append(f'<text x="{x}" y="{H-35}" text-anchor="middle">K={k}</text>')
    s.append(f'<polyline points="{pts1}" fill="none" stroke="#2563eb" stroke-width="3"/>')
    s.append(f'<polyline points="{pts2}" fill="none" stroke="#dc2626" stroke-width="3"/>')
    for k, v in zip(ks, y1):
        x, y = sx(k), sy1(v)
        s.append(f'<circle cx="{x}" cy="{y}" r="5" fill="#2563eb"/><text x="{x}" y="{y-10}" text-anchor="middle">{v:.2f}</text>')
    for k, v in zip(ks, y2):
        x, y = sx(k), sy2(v)
        s.append(f'<circle cx="{x}" cy="{y}" r="5" fill="#dc2626"/><text x="{x}" y="{y+20}" text-anchor="middle">{v:.3f}</text>')
    s.append(f'<text x="{m["l"]}" y="{H-12}">藍線：每位客戶群內差異，越低越好</text>')
    s.append(f'<text x="{W-m["r"]}" y="{H-12}" text-anchor="end">紅線：抽樣輪廓係數，越高越好</text>')
    s.append('</svg>')
    path.write_text('\n'.join(s), encoding='utf-8')


def color_for_z(z):
    z = max(-2, min(2, float(z)))
    if z >= 0:
        a = z / 2
        r = int(245 * (1 - a) + 220 * a)
        g = int(245 * (1 - a) + 38 * a)
        b = int(245 * (1 - a) + 38 * a)
    else:
        a = -z / 2
        r = int(245 * (1 - a) + 37 * a)
        g = int(245 * (1 - a) + 99 * a)
        b = int(245 * (1 - a) + 235 * a)
    return f'#{r:02x}{g:02x}{b:02x}'


def svg_heatmap(heat, path):
    heat_features = list(heat.columns)
    rows = list(heat.index)
    cellW, cellH = 95, 42
    left, top = 180, 90
    W = left + cellW * len(heat_features) + 30
    H = top + cellH * len(rows) + 80
    s = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">', '<rect width="100%" height="100%" fill="white"/>']
    s.append('<style>text{font-family:-apple-system,BlinkMacSystemFont,"Noto Sans TC",Arial,sans-serif;font-size:12px;fill:#222}.title{font-size:20px;font-weight:700}.small{font-size:11px}</style>')
    s.append(f'<text x="{W/2}" y="28" text-anchor="middle" class="title">K=5 客群標準化特徵熱圖</text>')
    s.append(f'<text x="{W/2}" y="50" text-anchor="middle">紅色代表高於重複投保者平均，藍色代表低於平均</text>')
    for j, c in enumerate(heat_features):
        x = left + j * cellW + cellW / 2
        label = feature_labels[c]
        s.append(f'<text x="{x}" y="{top-12}" text-anchor="middle" class="small">{html.escape(label)}</text>')
    for i, r in enumerate(rows):
        y = top + i * cellH
        s.append(f'<text x="{left-8}" y="{y+cellH/2+4}" text-anchor="end">{html.escape(r)}</text>')
        for j, c in enumerate(heat_features):
            x = left + j * cellW
            z = heat.loc[r, c]
            s.append(f'<rect x="{x}" y="{y}" width="{cellW}" height="{cellH}" fill="{color_for_z(z)}" stroke="white"/>')
            s.append(f'<text x="{x+cellW/2}" y="{y+cellH/2+4}" text-anchor="middle" fill="#111">{z:.1f}</text>')
    s.append(f'<text x="{W/2}" y="{H-18}" text-anchor="middle">數值為標準化後的群平均；非原始單位。</text>')
    s.append('</svg>')
    path.write_text('\n'.join(s), encoding='utf-8')


def svg_curves(curve_df, path):
    W, H = 920, 520
    m = {'l': 80, 'r': 30, 't': 50, 'b': 90}
    plotW = W - m['l'] - m['r']
    plotH = H - m['t'] - m['b']
    times = [30, 60, 120, 180, 365]
    colors = {
        '一次性/尚未回購客群': '#9ca3af',
        '高頻短週期多目的地型': '#dc2626',
        '一般亞洲穩定回購型': '#2563eb',
        '低頻長週期日本型': '#7c3aed',
        '長程高保費低頻型': '#059669',
        '國內短天期低保費型': '#d97706',
    }

    def sx(t):
        return m['l'] + (t - min(times)) / (max(times) - min(times)) * plotW

    def sy(v):
        return m['t'] + (100 - v) / 100 * plotH

    s = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}" height="{H}" viewBox="0 0 {W} {H}">', '<rect width="100%" height="100%" fill="white"/>']
    s.append('<style>text{font-family:-apple-system,BlinkMacSystemFont,"Noto Sans TC",Arial,sans-serif;font-size:12px;fill:#222}.title{font-size:20px;font-weight:700}.axis{stroke:#444;stroke-width:1}.grid{stroke:#e5e7eb;stroke-width:1}</style>')
    s.append(f'<text x="{W/2}" y="28" text-anchor="middle" class="title">各客群第一次到第二次投保累積回購率</text>')
    for v in [0, 20, 40, 60, 80, 100]:
        y = sy(v)
        s.append(f'<line x1="{m["l"]}" y1="{y}" x2="{W-m["r"]}" y2="{y}" class="grid"/>')
        s.append(f'<text x="{m["l"]-10}" y="{y+4}" text-anchor="end">{v}%</text>')
    s.append(f'<line x1="{m["l"]}" y1="{m["t"]}" x2="{m["l"]}" y2="{H-m["b"]}" class="axis"/>')
    s.append(f'<line x1="{m["l"]}" y1="{H-m["b"]}" x2="{W-m["r"]}" y2="{H-m["b"]}" class="axis"/>')
    for t in times:
        x = sx(t)
        s.append(f'<line x1="{x}" y1="{H-m["b"]}" x2="{x}" y2="{H-m["b"]+5}" stroke="#444"/>')
        s.append(f'<text x="{x}" y="{H-m["b"]+22}" text-anchor="middle">{t}天</text>')
    for g in cluster_order:
        vals = [curve_df.loc[g, f'{t}天'] for t in times]
        pts = ' '.join(f'{sx(t):.1f},{sy(v):.1f}' for t, v in zip(times, vals))
        s.append(f'<polyline points="{pts}" fill="none" stroke="{colors[g]}" stroke-width="3" opacity="0.9"/>')
        for t, v in zip(times, vals):
            s.append(f'<circle cx="{sx(t)}" cy="{sy(v)}" r="3.5" fill="{colors[g]}"/>')
    lx, ly, colw = m['l'], H - 50, 290
    for i, g in enumerate(cluster_order):
        x = lx + (i % 3) * colw
        y = ly + (i // 3) * 22
        s.append(f'<line x1="{x}" y1="{y}" x2="{x+24}" y2="{y}" stroke="{colors[g]}" stroke-width="4"/>')
        s.append(f'<text x="{x+30}" y="{y+4}">{html.escape(g)}</text>')
    s.append(f'<text x="{W/2}" y="{H-10}" text-anchor="middle">註：K-means 群組只含已觀察到重複投保者；一次性客群為 0% 是定義結果。</text>')
    s.append('</svg>')
    path.write_text('\n'.join(s), encoding='utf-8')

# 建立熱圖資料
zdf = pd.DataFrame(X, index=rep.index, columns=features)
zdf['cluster_name'] = rep['cluster_name']
heat_features = ['policies', 'avg_gap', 'avg_prem', 'avg_days', 'unique_dest', 'japan_share', 'domestic_share', 'longhaul_share', 'gap_le60_share', 'noncard_share']
heat = zdf.groupby('cluster_name')[heat_features].mean().reindex(cluster_order[1:])

svg_line_chart_k(k_table, OUT_DIR / 'K值選擇圖.svg')
svg_heatmap(heat, OUT_DIR / '客群特徵熱圖.svg')
svg_curves(curve, OUT_DIR / '客群回購曲線.svg')

print('已輸出：')
print(OUT_DIR / 'K值選擇圖.svg')
print(OUT_DIR / '客群特徵熱圖.svg')
print(OUT_DIR / '客群回購曲線.svg')

## 11. 各客群提醒策略表

以下是把分群與回購速度轉成商務操作的初步版本。  
這些策略不是最終結論，而是拿來跟助教確認：這樣從模型結果轉成行銷策略是否合理。

In [ ]:
strategy = pd.DataFrame([
    {
        '客群': '高頻短週期多目的地型',
        '追蹤優先級': '最高',
        '判斷依據': '平均投保 19.0 次、平均回購間隔 54.4 天、目的地數 7.35',
        '建議提醒時機': '30-45 天第一波，60 天左右第二波',
        '策略說明': '高頻、高多樣性，適合會員經營、旅遊提醒、目的地推薦',
    },
    {
        '客群': '一般亞洲穩定回購型',
        '追蹤優先級': '高',
        '判斷依據': '客群最大 15,997 人、平均回購間隔 124.5 天、保費貢獻高',
        '建議提醒時機': '60-90 天暖身，120-180 天主提醒',
        '策略說明': '規模大且穩定，是最適合主行銷的基礎客群',
    },
    {
        '客群': '低頻長週期日本型',
        '追蹤優先級': '中',
        '判斷依據': '日本占比 45.3%，但平均回購間隔 329.0 天',
        '建議提醒時機': '180 天後開始，接近 270-365 天加強',
        '策略說明': '適合日本旅遊季節、櫻花/楓葉/連假主題，不適合太密集提醒',
    },
    {
        '客群': '長程高保費低頻型',
        '追蹤優先級': '中高',
        '判斷依據': '平均保費 1,267 元、平均天數 21.0 天、長程占比 20.4%',
        '建議提醒時機': '90-120 天後再啟動，180-270 天追蹤',
        '策略說明': '單次價值高，但回購慢，適合長假、歐美旅遊與高保障方案',
    },
    {
        '客群': '國內短天期低保費型',
        '追蹤優先級': '中',
        '判斷依據': '國內占比 54.3%，平均保費低、平均天數 4.3 天',
        '建議提醒時機': '45-60 天第一波，120 天主提醒',
        '策略說明': '頻率可經營，但單次價值低，適合低成本自動化提醒',
    },
    {
        '客群': '一次性/尚未回購客群',
        '追蹤優先級': '待評估',
        '判斷依據': '只觀察到一筆，沒有回購間隔',
        '建議提醒時機': '不建議密集提醒，可用節慶/連假廣泛喚醒',
        '策略說明': '可能包含真正低活躍，也可能只是資料截止前尚未回購，需確認處理方式',
    },
])

strategy

## 12. 推薦主行銷對象

### 第一優先：高頻短週期多目的地型

這群客戶投保頻率最高、回購週期最短、目的地最多元。  
建議作為會員經營、個人化提醒與目的地推薦的核心對象。

### 第二優先：一般亞洲穩定回購型

這群人數最大，保費貢獻也高。  
建議作為主力行銷客群，以 60-180 天為主要提醒週期。

### 第三優先：長程高保費低頻型

這群客戶單次保費高、旅遊天數長。  
適合高保障、長程旅遊、申根/美澳等場景溝通，但不建議太早或太頻繁提醒。

### 條件式經營：低頻長週期日本型與國內短天期低保費型

低頻長週期日本型適合搭配日本旅遊旺季、櫻花季、楓葉季與連假提醒。  
國內短天期低保費型單次價值低，但可用低成本自動化方式維繫。

## 13. 需要與助教確認的假設與風險

這一節是明天討論的重點。

In [ ]:
assumptions = pd.DataFrame([
    {
        '假設/方法': 'K=5 是否合適',
        '目前跑出來的狀況': '輪廓係數 K=3 最高，但 K=5 更容易拆出可解釋客群',
        '建議諮詢助教的問題': '是否接受用商務可解釋性選 K=5，或應回到 K=3/K=4？',
    },
    {
        '假設/方法': '一次性客群獨立處理',
        '目前跑出來的狀況': '避免沒有回購間隔造成 K-means 扭曲',
        '建議諮詢助教的問題': '一次性客群是否應獨立，或需用右設限方法納入？',
    },
    {
        '假設/方法': '分群特徵包含回購間隔',
        '目前跑出來的狀況': '能分出回購節奏，但後續回購曲線會有循環解釋風險',
        '建議諮詢助教的問題': '是否同意將回購曲線定位為策略轉換，而非因果驗證？',
    },
    {
        '假設/方法': 'K-means 分群強度',
        '目前跑出來的狀況': '抽樣輪廓係數約 0.099，群與群不是非常分明',
        '建議諮詢助教的問題': '是否要減少特徵、改 RFM 分數法，或改階層式分群？',
    },
    {
        '假設/方法': 'CARD_BANK',
        '目前跑出來的狀況': '本次只用 999999 非信用卡比例，未用推估銀行名稱',
        '建議諮詢助教的問題': '是否保留在模型中，或只放在分群後描述？',
    },
    {
        '假設/方法': '特徵轉換',
        '目前跑出來的狀況': '對偏態數值做 log1p，並全部標準化',
        '建議諮詢助教的問題': '是否需要在正式報告中說明此步驟？',
    },
])

assumptions

## 14. 暫定結論

這版結果顯示，使用「K-means + 旅平險版 RFM 特徵」可以產出可解釋的客群，也能接上回購曲線與提醒策略。

但需要注意：

1. K-means 的分群品質指標不算非常強。
2. K=5 是基於商務可解釋性暫定採用，不是最終定論。
3. 分群特徵包含回購間隔，所以後續回購曲線應解讀為策略輔助，不應宣稱因果或模型驗證。
4. 一次性客戶獨立處理是合理但需要確認的假設。

明天建議優先問助教：

1. K=5 是否可接受？
2. 是否要減少特徵或改用 RFM 分數法？
3. 一次性客戶處理方式是否合理？
4. 回購曲線的解釋方式是否足夠嚴謹？